In [3]:
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, r2_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

SEED = 2026

np.random.seed(SEED)
rng = np.random.default_rng(SEED)

In [4]:
df = pd.read_csv("hris_performance_train.csv")
df.head()

,department_code,age_years,tenure_months,tenure_years,overtime_hours_month,remote_work_ratio,job_level,goal_completion_rate,engagement_score,job_satisfaction,absences_days_12mo,engagement_percent,commute_minutes,training_hours_12mo,team_size,last_promotion_years_ago,performance_rating
0,6,19,6,0.576,18,0.322,6,0.669,94.730,93.035,0.274,0.947,5,8,12,0,2
1,6,53,50,3.893,2,0.807,5,0.109,70.053,71.523,6.313,0.688,45,14,11,2,4
2,7,26,7,0.486,19,0.620,5,0.698,66.642,49.943,4.203,0.666,87,5,6,1,3
3,1,29,64,5.386,2,0.563,3,0.256,48.418,87.839,0.347,0.492,43,24,25,0,4
4,3,50,23,2.095,6,0.819,5,0.855,50.685,52.825,0.100,0.504,78,31,15,4,3


In [5]:
df.describe()

,department_code,age_years,tenure_months,tenure_years,overtime_hours_month,remote_work_ratio,job_level,goal_completion_rate,engagement_score,job_satisfaction,absences_days_12mo,engagement_percent,commute_minutes,training_hours_12mo,team_size,last_promotion_years_ago,performance_rating
count,97500.000000,97500.000000,97500.000000,97500.000000,97500.000000,97500.00000,97500.000000,97500.000000,97500.000000,97500.000000,97500.000000,97500.000000,97500.000000,97500.000000,97500.000000,97500.000000,97500.000000
mean,6.472513,41.652431,42.800985,3.564971,8.814626,0.49922,4.732000,0.480639,39.616620,50.439046,2.881423,0.396207,62.579918,17.767815,13.475600,2.201221,3.036892
std,2.342370,9.946456,58.980924,4.918540,10.432723,0.20828,1.469566,0.249976,24.473869,26.450715,4.667755,0.244914,24.276504,20.924172,4.869767,2.649695,0.898210
min,1.000000,18.000000,0.000000,0.000000,0.000000,0.01400,1.000000,0.003000,0.113000,1.118000,0.000000,0.000000,5.000000,0.000000,2.000000,0.000000,1.000000
25%,5.000000,35.000000,8.000000,0.686000,2.000000,0.33800,4.000000,0.273000,19.219750,27.835750,0.467000,0.192000,46.000000,5.000000,10.000000,1.000000,3.000000
50%,6.000000,42.000000,23.000000,1.902000,5.000000,0.49900,5.000000,0.465000,36.065500,50.430000,1.255000,0.361000,63.000000,11.000000,13.000000,1.000000,3.000000
75%,8.000000,49.000000,52.000000,4.290000,11.000000,0.66100,6.000000,0.689000,57.467250,72.917000,3.165000,0.575000,79.000000,22.000000,17.000000,3.000000,3.000000
max,12.000000,65.000000,360.000000,30.000000,60.000000,0.98700,8.000000,0.998000,99.832000,99.950000,30.000000,1.000000,120.000000,120.000000,25.000000,15.000000,5.000000


In [6]:
y = df["performance_rating"]
X = df.drop(columns="performance_rating")

y_linear = df["job_satisfaction"]
X_linear = df.drop(columns="job_satisfaction")

In [7]:
X_model = X

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_model)

In [8]:
lm = LinearRegression(fit_intercept=True)
lm.fit(X_linear, y_linear)
y_hat = lm.predict(X_linear)
mse = np.mean((y_hat - y_linear)**2)
r2 = lm.score(X_linear, y_linear)
print(f"MSE = {mse:.3f} and r^2 = {r2:.3f}")

MSE = 109.722 and r^2 = 0.843


In [9]:
lr = LogisticRegression(
    penalty=None, C=0.07, fit_intercept=True, random_state=SEED, max_iter=10_000, solver="lbfgs"
)
lr.fit(X_model, y)

lr.score(X_model, y)

/home/ari/repos/psyc_723/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


0.575774358974359

In [10]:
knn = KNeighborsClassifier()
knn.fit(X_scaled, y)

knn.score(X_scaled, y)

0.7075692307692307

In [11]:
rf = RandomForestClassifier(n_estimators=100, random_state=SEED, min_samples_split=15)

rf.fit(X_model, y)
rf.score(X_model, y)

0.8736923076923077

In [12]:
X.columns[[3, 6, 7, 8, 10]]

Index(['tenure_years', 'job_level', 'goal_completion_rate', 'engagement_score',
       'absences_days_12mo'],
      dtype='object')

In [21]:
test = pd.read_csv("hris_performance_hidden_test.csv")

y_test = test["performance_rating"]
X_test = test.drop(columns="performance_rating")

y_lr_test = test["job_satisfaction"]
X_lr_test = test.drop(columns="job_satisfaction")

In [22]:
lm.score(X_lr_test, y_lr_test)

0.8441191028957097

In [14]:
lr.score(X_test, y_test)

0.5764615384615385

In [17]:
scaler = StandardScaler()
X_test_scaled = scaler.fit_transform(X_test)

knn.score(X_test_scaled, y_test)

0.6119076923076923

In [19]:
rf.score(X_test, y_test)

0.7441230769230769